In [1]:
import torch
from mario_gpt import MarioDataset, MarioLM, TrainingConfig, MarioGPTTrainer
from mario_gpt.utils import view_level, convert_level_to_png, join_list_of_list, characterize

In [2]:
mario_lm = MarioLM()
dataset = MarioDataset(tokenizer=mario_lm.tokenizer, folder_path='/mario-gpt/mario_gpt/levels')

Using shyamsn97/Mario-GPT2-700-context-length lm


/opt/conda/lib/python3.10/site-packages/transformers-4.46.3-py3.10.egg/transformers/models/auto/modeling_auto.py:1833: FutureWarning: The class `AutoModelWithLMHead` is deprecated and will be removed in a future version. Please use `AutoModelForCausalLM` for causal language models, `AutoModelForMaskedLM` for masked language models and `AutoModelForSeq2SeqLM` for encoder-decoder models.
  warnings.warn(


Using shyamsn97/Mario-GPT2-700-context-length tokenizer


Token indices sequence length is longer than the specified maximum sequence length for this model (1540 > 1024). Running this sequence through the model will result in indexing errors


In [3]:
# view_level(dataset.data[1]["input_ids"][:700], mario_lm.tokenizer)
view_level(dataset.input_ids[:700], mario_lm.tokenizer)

['SSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSS',
 'S-------------------------------------------------',
 'S-------------------------------------------------',
 'S-------------------------------------------------',
 'S-------------------------------------------------',
 'S-------------------------------------------------',
 'S---------------------------------oo--------------',
 'S-------------------------------------------------',
 'S---------------!S?S!-----------------------------',
 'S----------------------------------------------<>-',
 'S----------------------SS---------SS------()---[]-',
 'S---------g--g---------SS-----k---SS------[]---[]-',
 'XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX--XXXXXX[]-',
 'XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX--XXXXXX[]-']

In [4]:
!pip -q install --upgrade huggingface_hub 
!apt -q install git -y
!pip -q install groq
!pip -q install python-dotenv

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Reading package lists...
Building dependency tree...
Reading state information...
git is already the newest version (1:2.17.1-1ubuntu0.18).
0 upgraded, 0 newly installed, 0 to remove and 58 not upgraded.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [5]:
from mario_gpt.prompter import Prompter
from mario_gpt.prompt_adapter import PromptAdapter
from transformers import pipeline
from groq import Groq
from tqdm import tqdm
from transformers import AutoTokenizer
# from huggingface_hub import login
from dotenv import load_dotenv
import os
load_dotenv()

# Pre-initialize the LLM model
# login(token=os.getenv("HUGGINGFACE_TOKEN"))
# llm_model = pipeline("text-generation", model="meta-llama/Llama-3.1-8B", device=0)
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
tokenizer_path = "shyamsn97/Mario-GPT2-700-context-length"
level_tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
prompter = Prompter(level_tokenizer=level_tokenizer)
adapter = PromptAdapter(llm=client, prompter=prompter)

In [6]:
import json
import torch
from typing import Tuple

def get_item_from_dataset(dataset, indice: int) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Retrieves an item (input_ids and attention_mask) from the dataset at a given index.
    """
    input_ids, attention_mask = dataset[indice]
    return torch.stack([input_ids]), torch.stack([attention_mask])

def save_dataset_to_json(dataset, output_file: str):
    """
    Saves the first two indexed items from the dataset into a JSON file.
    """
    data_list = []
    # for i in range(2):  # Get first two items
    for i in tqdm(range(len(dataset)), desc="Processing dataset items"):
        input_ids, attention_masks = get_item_from_dataset(dataset, i)
        
        for level, mask in zip(input_ids, attention_masks):
            prompt_base, _, str_level = prompter(level=level)  
            adapted_prompt = adapter.adapt_prompt(prompt_base, True)
            # print(adapted_prompt)

            data_list.append({
                "input_ids": level.tolist(),  
                "attention_mask": mask.tolist(),
                "prompt": prompt_base,
                "adapted_prompt": adapted_prompt,
                "str_level": str_level
            })

    # Save to JSON file
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(data_list, f, indent=4)


In [7]:
save_dataset_to_json(dataset, "mario_dataset.json")

Processing dataset items: 100% 60/60 [03:30<00:00,  3.50s/it]
